In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit 
import os
import openpyxl
from pandas import *
%matplotlib inline

In [2]:
import pylab
params = {'backend': 'ps',
          'axes.labelsize': 20,
          'legend.fontsize': 20,
          'legend.handlelength': 2.0,
          'legend.borderaxespad': 0,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'font.family': 'serif',
          'font.size': 20,
          'font.weight': 'normal',
          #'font.serif': ['Palatino'],
          'ps.usedistiller': 'xpdf',
          'text.usetex': False,
         'agg.path.chunksize' : 1000}
pylab.rcParams.update(params)

In [3]:
dt = 1 / 24
nyears = 2000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Poisson
rain = md.poisson_process(n = int(nsim), alpha = 14.42/1000, lbd = (0.19 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 456.3

# Half sine wave shape over 365 days
shape = np.sin((np.pi*days)/365)                       # sinB(x-h) + K; B = pi/365, h = 0, K = 0

# Normalize so that the sum equals the annual NPP
daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [4]:
print("Maximum Value is:", max(npp))
print("Length is:", len(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 1.9637045245762004
Length is: 17520000
Average value is: 1.2501369863013678


In [5]:
total_rainfall = rain.sum()
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year
print(f"Average rainfall per year = {RPY:.3f} mm/year")

Total rainfall over 450 years = 2002.933 meters
Average rainfall per year = 1001.466 mm/year


In [6]:
PET = 689.1
AI = PET/RPY
print(AI)

"""
def hill_constrained(R, KA, n):
    R_max = 4.18

    numerator = R**n
    denominator = KA**n + R**n
    norm_factor = R_max**n / (KA**n + R_max**n)
    TI = (numerator / denominator) / norm_factor

    # Clip to ensure TI does not exceed 1
    return (np.minimum(TI, 1.0))

TI = hill_constrained(AI, 3.45, 2.28)
print(TI)
"""

# Final best-fit parameters from your latest run
KA_final = 9.0132
n_final = 1.7444
R_max = 3.8  # Max AI used for normalization

def compute_TI(AI):
    AI = np.asarray(AI)  # Accept scalars or arrays
    numerator = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor
    return np.minimum(TI, 1.0)

TI = compute_TI(AI)
print("TI =", TI)

0.6880910131416331
TI = 0.061306527019946655


In [7]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.1   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [8]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████| 17520000/17520000 [49:24<00:00, 5910.89it/s]  


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       883.642328   89.317698   30.430647  0.080104  0.002334  0.210862   
2       979.655067   43.410457   29.578815  0.088008  0.002085  0.102484   
3      1074.606289   21.238611   31.071678  0.095871  0.001649  0.050140   
4      1163.189535   10.478284   32.450708  0.104941  0.001171  0.024737   
5      1246.330427    5.218602   34.318903  0.112897  0.000740  0.012320   
...            ...         ...         ...       ...       ...       ...   
1996   1742.203625  129.960032  205.463478  0.157894  0.004955  0.306786   
1997   1742.233503  131.432212  204.221332  0.159082  0.005001  0.310260   
1998   1742.163143  131.170708  198.725812  0.158498  0.004981  0.309644   
1999   1742.040822  129.615097  204.920339  0.160959  0.005049  0.305972   
2000   1741.863777  129.154637  208.826223  0.158676  0.004974  0.304884   

           

In [9]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Poirier\POI.csv", index= True, header = True)

sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\POI.csv", index= True, header = True)